## Semantic Search (vector)

In [3]:
from qdrant_client import QdrantClient, models

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-05 02:50:23.896562499 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [4]:
client=QdrantClient('http://127.0.0.1:6333')

In [5]:
import requests

docs_url='https://github.com/alexeygrigorev/llm-rag-workshop/raw/main/notebooks/documents.json'
docs_response=requests.get(docs_url)
documents_raw=docs_response.json()

In [9]:
documents_raw[0]['documents'][:5]

[{'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
  'section': 'General course-related questions',
  'question': 'Course - When will the course start?'},
 {'text': 'GitHub - DataTalksClub data-engineering-zoomcamp#prerequisites',
  'section': 'General course-related questions',
  'question': 'Course - What are the prerequisites for this course?'},
 {'text': "Yes, even if you don't register, you're still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute."

### Basic Architecture

Text  <strong>----></strong>  Embedding Model <strong>--(turn it to)--></strong> vector <strong>--(store it to)--></strong> vector DB (Qdrant) <br>
Questions  <strong>----></strong>  Embedding Model <strong>--(turn it to)--></strong> vector <strong>--(compare it to)--></strong> vector DB (Qdrant)


#### Choosing Embedding model

In [10]:
from fastembed import TextEmbedding
TextEmbedding.list_supported_models()

[{'model': 'BAAI/bge-base-en',
  'sources': {'hf': 'Qdrant/fast-bge-base-en',
   'url': 'https://storage.googleapis.com/qdrant-fastembed/fast-bge-base-en.tar.gz',
   '_deprecated_tar_struct': True},
  'model_file': 'model_optimized.onnx',
  'description': 'Text embeddings, Unimodal (text), English, 512 input tokens truncation, Prefixes for queries/documents: necessary, 2023 year.',
  'license': 'mit',
  'size_in_GB': 0.42,
  'additional_files': [],
  'dim': 768,
  'tasks': {}},
 {'model': 'BAAI/bge-base-en-v1.5',
  'sources': {'hf': 'qdrant/bge-base-en-v1.5-onnx-q',
   'url': 'https://storage.googleapis.com/qdrant-fastembed/fast-bge-base-en-v1.5.tar.gz',
   '_deprecated_tar_struct': True},
  'model_file': 'model_optimized.onnx',
  'description': 'Text embeddings, Unimodal (text), English, 512 input tokens truncation, Prefixes for queries/documents: not so necessary, 2023 year.',
  'license': 'mit',
  'size_in_GB': 0.21,
  'additional_files': [],
  'dim': 768,
  'tasks': {}},
 {'model':

In [17]:
#for this use case, we use small-to-moderate sized embeddings (512 dimensions)
import json

for i in TextEmbedding.list_supported_models():
    if i['dim']==512:
        print(json.dumps(i, indent=2))

{
  "model": "BAAI/bge-small-zh-v1.5",
  "sources": {
    "hf": "Qdrant/bge-small-zh-v1.5",
    "url": "https://storage.googleapis.com/qdrant-fastembed/fast-bge-small-zh-v1.5.tar.gz",
    "_deprecated_tar_struct": true
  },
  "model_file": "model_optimized.onnx",
  "description": "Text embeddings, Unimodal (text), Chinese, 512 input tokens truncation, Prefixes for queries/documents: not so necessary, 2023 year.",
  "license": "mit",
  "size_in_GB": 0.09,
  "additional_files": [],
  "dim": 512,
  "tasks": {}
}
{
  "model": "Qdrant/clip-ViT-B-32-text",
  "sources": {
    "hf": "Qdrant/clip-ViT-B-32-text",
    "url": null,
    "_deprecated_tar_struct": false
  },
  "model_file": "model.onnx",
  "description": "Text embeddings, Multimodal (text&image), English, 77 input tokens truncation, Prefixes for queries/documents: not necessary, 2021 year",
  "license": "mit",
  "size_in_GB": 0.25,
  "additional_files": [],
  "dim": 512,
  "tasks": {}
}
{
  "model": "jinaai/jina-embeddings-v2-small-e

We will use jina-embeddings-v2-small-en (english, unimodal/text)

In [20]:
model_handle="jinaai/jina-embeddings-v2-small-en"

#### Create Collection

For context:
* <b>Points</b> are the cental entity Qdrant works with. <br>
  A point is a record consisting odf an ID, a vector, and optional payload/metadata
* A <b>collection</b> is a named set of points that we can search within <br>
  <i> Think of it as the container for your vector search solution, a single business problem solved</i>

In [18]:
collection_name='zoomcamp-rag'

In [19]:
client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=512, #vectors dimensionality
        distance=models.Distance.COSINE #distance metric for similarity search
    )
)

True

#### Create, Embed & Insert Points into Collection

In [27]:
for course in documents_raw[:1]:
    for doc in course['documents'][:5]:
        print(course['course'])

data-engineering-zoomcamp
data-engineering-zoomcamp
data-engineering-zoomcamp
data-engineering-zoomcamp
data-engineering-zoomcamp


In [29]:
points=[]
id = 0 

for course in documents_raw:
    for doc in course['documents']:
        point=models.PointStruct(
            id=id,
            vector=models.Document(text=doc['text'], model=model_handle),
            payload={
                "text":doc['text'],
                "section": doc['section'],
                "course": course['course']
            }
        )
        points.append(point)
        id+=1

#### Upsert to Qdrant

In [31]:
client.upsert(
    collection_name=collection_name,
    points=points
)

Fetching 5 files: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:03<00:00,  1.30it/s]


UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

#### Running Semantic Similarity Search

How it works:<br>

1. Qdrant compares query vector to stored vector (based on vector index) using distance metric defined when creating the collection.
2. The closest matches returned ranked by similarity

In [38]:
def search(query, limit=1):
    results=client.query_points(
        collection_name=collection_name,
        query=models.Document( #embed the query with model
            text=query,
            model=model_handle
        ),
        limit = limit, #limit to 1 
        with_payload=True #get metadata in the results
    )
    return results

In [34]:
import random

course=random.choice(documents_raw)
course_piece=random.choice(course['documents'])
print(json.dumps(course_piece, indent=1))

{
 "text": "The Root Mean Squared Error (RMSE) is one of the primary metrics to evaluate the performance of a regression model. It calculates the average deviation between the model's predicted values and the actual observed values, offering insight into the model's ability to accurately forecast the target variable. To calculate RMSE score:\nLibraries needed\nimport numpy as np\nfrom sklearn.metrics import mean_squared_error\nmse = mean_squared_error(actual_values, predicted_values)\nrmse = np.sqrt(mse)\nprint(\"Root Mean Squared Error (RMSE):\", rmse)\n(Aminat Abolade)",
 "section": "2. Machine Learning for Regression",
 "question": "Understanding RMSE and how to calculate RMSE score"
}


In [54]:
res=search(course_piece['question'])

In [63]:
res.points[0].payload['text']

'The Root Mean Squared Error (RMSE) is one of the primary metrics to evaluate the performance of a regression model. It calculates the average deviation between the model\'s predicted values and the actual observed values, offering insight into the model\'s ability to accurately forecast the target variable. To calculate RMSE score:\nLibraries needed\nimport numpy as np\nfrom sklearn.metrics import mean_squared_error\nmse = mean_squared_error(actual_values, predicted_values)\nrmse = np.sqrt(mse)\nprint("Root Mean Squared Error (RMSE):", rmse)\n(Aminat Abolade)'

In [65]:
print(search('how much time should i allocate per weeks for the ml engineering course'))

points=[ScoredPoint(id=10, version=1, score=0.8229809, payload={'text': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week. [source1] [source2]\nYou can also calculate it yourself using this data and then update this answer.', 'section': 'General course-related questions', 'course': 'data-engineering-zoomcamp'}, vector=None, shard_key=None, order_value=None)]


In [73]:
documents_raw[0]['documents'][:5]

[{'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
  'section': 'General course-related questions',
  'question': 'Course - When will the course start?'},
 {'text': 'GitHub - DataTalksClub data-engineering-zoomcamp#prerequisites',
  'section': 'General course-related questions',
  'question': 'Course - What are the prerequisites for this course?'},
 {'text': "Yes, even if you don't register, you're still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute."

#### Running Similarity Search with Filters

First we create indexing on "course"

In [74]:
client.create_payload_index(
    collection_name=collection_name,
    field_name='course',
    field_schema='keyword'
)

UpdateResult(operation_id=3, status=<UpdateStatus.COMPLETED: 'completed'>)

Then we tell Qdrant to lookup:<br>
1. key="course" from payload
2. match the "course" with value=course (from function)

Refresher on how point looks like

In [81]:
points[0].payload

{'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
 'section': 'General course-related questions',
 'course': 'data-engineering-zoomcamp'}

In [82]:
points[0].payload["course"]

'data-engineering-zoomcamp'

In [85]:
def search_in_course(query, course, limit=1):

    results = client.query_points(
        collection_name=collection_name,
        query=models.Document( #embed the query text locally with "jinaai/jina-embeddings-v2-small-en"
            text=query,
            model=model_handle
        ),
        query_filter=models.Filter( # filter by course name
            must=[
                models.FieldCondition(
                    key="course",
                    match=models.MatchValue(value=course)
                )
            ]
        ),
        limit=limit, # top closest matches
        with_payload=True #to get metadata in the results
    )

    return results

In [88]:
print(search_in_course('how much time should i allocate per weeks for the course','machine-learning-zoomcamp'))

points=[ScoredPoint(id=442, version=1, score=0.8382912, payload={'text': 'Around ~10 hours per week. Timur Kamaliev did a detailed analysis of how much time students of the previous cohort needed to spend on different modules and projects. Full article', 'section': 'General course-related questions', 'course': 'machine-learning-zoomcamp'}, vector=None, shard_key=None, order_value=None)]
